## Traffic Management

In modernen Cloud-Umgebungen setzen viele Unternehmen auf Microservices-Architekturen, um Skalierbarkeit, Flexibilität und Wartbarkeit zu verbessern. Dabei entstehen jedoch Herausforderungen in Bezug auf Kommunikation, Sicherheit und Monitoring zwischen den Services. Hier kommt **Istio** ins Spiel – eine leistungsstarke Service-Mesh-Plattform, die speziell entwickelt wurde, um die Verwaltung und Steuerung von Microservices zu erleichtern.

Istio bietet eine transparente Schicht zwischen den Services, die **Traffic-Management, Sicherheitsrichtlinien, Observability** und weitere Funktionen bereitstellt, ohne dass die Anwendungslogik angepasst werden muss. Durch den Einsatz eines **Sidecar-Proxys** (wie **Envoy**) kann Istio den Datenverkehr innerhalb eines Clusters überwachen, steuern und absichern.

### Installation (Version 1.2x mit Zipkin)

Diese Variante braucht weniger Ressourcen und wird auch mit Kubernetes >= 1.32 unterstützt.

Statt Kiali und Jaeger kommt Zipkin zum Einsatz, welcher ungefähr den gleichen Nutzen bietet.

**Installation Istio Sourcen und CLI**

In [ ]:
%%bash
export ISTIO_VERSION=1.24.2 
curl -L https://istio.io/downloadIstio | sh -
sudo cp istio-${ISTIO_VERSION}/bin/istioctl /usr/local/bin/

### Zipkin

Zipkin ist ein verteiltes tracing system. Es hilft beim Sammeln von Zeitdaten, die zur Behebung von Latenzproblemen in Service-Mesh erforderlich sind.  

- - -

Minimale Istio Installation mit Zipkin.

In [ ]:
%%bash

cat <<EOF > ./tracing.yaml
apiVersion: install.istio.io/v1alpha1
kind: IstioOperator
spec:
  meshConfig:
    enableTracing: true
    defaultConfig:
      tracing:
        sampling: 0.1  # Nur 10% aller Anfragen werden getraced
      proxyMetadata:
        ISTIO_META_ENABLE_ACCESS_LOG: "false"  # Deaktiviert Access-Logs (optional)        
    extensionProviders:
    - name: zipkin
      zipkin:
        service: zipkin.istio-system.svc.cluster.local
        port: 9411
EOF
istioctl install -f ./tracing.yaml --skip-confirmation

kubectl apply -f - <<EOF
apiVersion: telemetry.istio.io/v1
kind: Telemetry
metadata:
  name: mesh-default
  namespace: istio-system
spec:
  tracing:
  - providers:
    - name: zipkin
EOF



Installation von Zipkin

In [ ]:
%%bash
kubectl apply -f https://raw.githubusercontent.com/istio/istio/release-1.24/samples/addons/extras/zipkin.yaml

UI Port von Zipkin weiterleiten

In [ ]:
%%bash
kubectl get service -n istio-system -l name=zipkin -o yaml | sed 's/ClusterIP/NodePort/g' | kubectl apply -f -
echo "Zipkin  UI: http://"$(cat ~/work/server-ip)":"$(kubectl get -n istio-system service/zipkin -o jsonpath='{.spec.ports[?(@.name=="http-query")].nodePort}')

Damit ist die Installation von Istio abgeschlossen und wir haben ein leichtgewichtiges Traffic Management ohne Prometheus, Grafana, Kiali und Jaeger.


In [ ]:
%%bash
kubectl get pods -n istio-system

Weiter geht es mit den Service Mesh Beispielen:
* [Einfacher Service Mesh](01-service-mesh.ipynb)

---

### Aufräumen

Wen Istio nicht mehr benötigt wird, kann es entfernt werden.

In [ ]:
%%bash
export ISTIO_VERSION=1.24.2
kubectl delete -f https://raw.githubusercontent.com/istio/istio/release-1.24/samples/addons/extras/zipkin.yaml
istioctl uninstall -y --purge